# 07 - Full Dataset Pipeline (Week 1)

**Goal:** Parse all 50 CholecT50 video JSONs, build graphs for every video, generate dataset-wide statistics, create temporal graph sequences for downstream modeling, and establish train/val/test splits.

**Builds on:** Notebooks 01-06 (pilot with 3 videos)
**Output:** Full dataset ready for model training (Week 2+)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from pathlib import Path
import json
import time
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from src.triplet_parser import (
    parse_video, filter_valid_triplets, video_summary,
    save_parsed_csv, load_categories, multi_label_analysis,
)
from src.graph_builder import build_graph, save_graph, graph_stats

# Paths
LABELS_DIR = Path("/Users/maheshkundurthi/Documents/Research work OPT/CholecT50/labels")
PROJECT_ROOT = Path("..").resolve()
OUTPUT_DIR = PROJECT_ROOT / "outputs"
CSV_DIR = OUTPUT_DIR / "parsed_triplets"
GRAPH_DIR = OUTPUT_DIR / "graphs"
STATS_DIR = OUTPUT_DIR / "stats"
SEQ_DIR = OUTPUT_DIR / "temporal_sequences"

for d in [CSV_DIR, GRAPH_DIR, STATS_DIR, SEQ_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Labels dir: {LABELS_DIR}")
print(f"Output dir: {OUTPUT_DIR}")
json_files = sorted(LABELS_DIR.glob("VID*.json"))
print(f"Found {len(json_files)} video JSONs")

## 1. Parse All 50 CholecT50 Videos

Run every video JSON through `triplet_parser.parse_video()`, filter to valid triplets, and save per-video CSVs plus a combined CSV.

In [ ]:
all_dfs = []
parse_results = []

for json_path in json_files:
    vid_name = json_path.stem
    t0 = time.time()
    
    df_raw = parse_video(json_path)
    df_valid = filter_valid_triplets(df_raw)
    elapsed = time.time() - t0
    
    # Save per-video CSV
    csv_path = CSV_DIR / f"{vid_name}_triplets.csv"
    df_valid.to_csv(csv_path, index=False)
    
    all_dfs.append(df_valid)
    parse_results.append({
        "video": vid_name,
        "raw_rows": len(df_raw),
        "valid_rows": len(df_valid),
        "frames": df_valid["frame"].nunique(),
        "unique_triplets": df_valid["triplet_id"].nunique(),
        "time_sec": round(elapsed, 2),
    })

df_all = pd.concat(all_dfs, ignore_index=True)
df_all.to_csv(CSV_DIR / "all_triplets.csv", index=False)

df_parse = pd.DataFrame(parse_results)
print(f"Parsed {len(df_parse)} videos, {len(df_all)} total valid rows")
df_parse

## 2. Build Graphs for All Videos

Construct a NetworkX `MultiDiGraph` for every video and save as GEXF.

In [ ]:
all_graph_stats = []

for vid_name, df_vid in df_all.groupby("video"):
    G = build_graph(df_vid, video_name=vid_name)
    save_graph(G, GRAPH_DIR / f"{vid_name}_graph.gexf")
    stats = graph_stats(G)
    all_graph_stats.append(stats)

df_gstats = pd.DataFrame(all_graph_stats)
df_gstats.to_csv(STATS_DIR / "graph_stats_all.csv", index=False)
print(f"Built {len(df_gstats)} graphs")
df_gstats[["video", "nodes", "n_instruments", "n_targets", "edges", "density", "total_frames"]]

## 3. Dataset-Wide Statistics

### 3.1 Global Summary

In [ ]:
summary = video_summary(df_all)
summary.to_csv(STATS_DIR / "video_summary_all.csv", index=False)

ml_stats = multi_label_analysis(df_all)

global_stats = {
    "total_videos": df_all["video"].nunique(),
    "total_valid_rows": len(df_all),
    "unique_triplet_classes": df_all["triplet_id"].nunique(),
    "unique_instruments": df_all["instrument"].nunique(),
    "unique_verbs": df_all["verb"].nunique(),
    "unique_targets": df_all["target"].nunique(),
    "instruments": sorted(df_all["instrument"].unique().tolist()),
    "verbs": sorted(df_all["verb"].unique().tolist()),
    "targets": sorted(df_all["target"].unique().tolist()),
    "multi_label_pct": ml_stats["multi_label_pct"],
    "max_triplets_per_frame": ml_stats["max_triplets_per_frame"],
    "mean_triplets_per_frame": ml_stats["mean_triplets_per_frame"],
}

with open(STATS_DIR / "global_dataset_stats.json", "w") as f:
    json.dump(global_stats, f, indent=2)

for k, v in global_stats.items():
    if not isinstance(v, list):
        print(f"  {k}: {v}")

### 3.2 Per-Video Frame Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Frames per video
summary_sorted = summary.sort_values("total_frames", ascending=True)
axes[0].barh(summary_sorted["video"], summary_sorted["total_frames"], color="#2196F3", alpha=0.8)
axes[0].set_xlabel("Frames")
axes[0].set_title("Frames per Video")
axes[0].tick_params(axis="y", labelsize=7)

# Triplet rows per video
axes[1].barh(summary_sorted["video"], summary_sorted["total_triplet_rows"], color="#4CAF50", alpha=0.8)
axes[1].set_xlabel("Triplet Rows")
axes[1].set_title("Valid Triplet Rows per Video")
axes[1].tick_params(axis="y", labelsize=7)

plt.tight_layout()
plt.savefig(STATS_DIR / "frames_per_video.png", dpi=120, bbox_inches="tight")
plt.show()

### 3.3 Triplet Class Distribution

In [ ]:
triplet_dist = df_all.groupby("triplet_label").size().sort_values(ascending=False)
triplet_dist.to_csv(STATS_DIR / "triplet_class_distribution.csv", header=["count"])

fig, ax = plt.subplots(figsize=(14, 6))
top_30 = triplet_dist.head(30)
ax.barh(range(len(top_30)), top_30.values, color="#FF7043", alpha=0.85)
ax.set_yticks(range(len(top_30)))
ax.set_yticklabels(top_30.index, fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("Frequency (total rows)")
ax.set_title("Top 30 Triplet Classes by Frequency")
plt.tight_layout()
plt.savefig(STATS_DIR / "triplet_class_distribution.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"Total triplet classes: {len(triplet_dist)}")
print(f"Top 5:\n{top_30.head()}")
print(f"\nBottom 5:\n{triplet_dist.tail()}")

### 3.4 Phase Distribution

In [ ]:
if "phase" in df_all.columns:
    phase_dist = df_all.groupby("phase").agg(
        videos=("video", "nunique"),
        frames=("frame", "nunique"),
        rows=("triplet_id", "count"),
    ).sort_values("rows", ascending=False)
    phase_dist.to_csv(STATS_DIR / "phase_distribution.csv")
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(phase_dist.index, phase_dist["rows"], color="#AB47BC", alpha=0.8)
    ax.set_xlabel("Total Triplet Rows")
    ax.set_title("Triplet Rows per Surgical Phase")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig(STATS_DIR / "phase_distribution.png", dpi=120, bbox_inches="tight")
    plt.show()
    
    phase_dist

### 3.5 Instrument and Verb Usage

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Instrument usage
inst_dist = df_all.groupby("instrument").size().sort_values(ascending=True)
axes[0].barh(inst_dist.index, inst_dist.values, color="#2196F3", alpha=0.8)
axes[0].set_title("Instrument Usage (total rows)")

# Verb usage
verb_dist = df_all.groupby("verb").size().sort_values(ascending=True)
axes[1].barh(verb_dist.index, verb_dist.values, color="#FF9800", alpha=0.8)
axes[1].set_title("Verb Usage (total rows)")

plt.tight_layout()
plt.savefig(STATS_DIR / "instrument_verb_distribution.png", dpi=120, bbox_inches="tight")
plt.show()

## 4. Temporal Graph Sequences

For each video, build a chronological sequence of graph states. Each frame gets:
- A **multi-hot vector** (100-d) over all triplet classes
- An **edge list** with instrument/verb/target indices
- The **surgical phase** label

This is the training data format for the models in Week 2+.

In [ ]:
# Build label mappings
all_triplet_labels = sorted(df_all["triplet_label"].unique().tolist())
triplet_to_idx = {label: idx for idx, label in enumerate(all_triplet_labels)}
num_classes = len(all_triplet_labels)

instrument_labels = sorted(df_all["instrument"].unique().tolist())
verb_labels = sorted(df_all["verb"].unique().tolist())
target_labels = sorted(df_all["target"].unique().tolist())

entity_mapping = {
    "instruments": {label: idx for idx, label in enumerate(instrument_labels)},
    "verbs": {label: idx for idx, label in enumerate(verb_labels)},
    "targets": {label: idx for idx, label in enumerate(target_labels)},
}

with open(SEQ_DIR / "triplet_label_mapping.json", "w") as f:
    json.dump(triplet_to_idx, f, indent=2)
with open(SEQ_DIR / "entity_label_mapping.json", "w") as f:
    json.dump(entity_mapping, f, indent=2)

print(f"Triplet classes: {num_classes}")
print(f"Instruments: {len(instrument_labels)} -> {instrument_labels}")
print(f"Verbs: {len(verb_labels)} -> {verb_labels}")
print(f"Targets: {len(target_labels)} -> {target_labels}")

In [ ]:
for vid_name, df_vid in df_all.groupby("video"):
    frames = sorted(df_vid["frame"].unique())
    sequence_data = []
    
    for frame_id in frames:
        frame_df = df_vid[df_vid["frame"] == frame_id]
        
        # Multi-hot encoding
        multi_hot = np.zeros(num_classes, dtype=np.int8)
        active_indices = []
        for label in frame_df["triplet_label"].unique():
            if label in triplet_to_idx:
                idx = triplet_to_idx[label]
                multi_hot[idx] = 1
                active_indices.append(idx)
        
        # Edge-level detail
        edges = []
        for _, row in frame_df.iterrows():
            edges.append({
                "instrument": row["instrument"],
                "instrument_idx": entity_mapping["instruments"].get(row["instrument"], -1),
                "verb": row["verb"],
                "verb_idx": entity_mapping["verbs"].get(row["verb"], -1),
                "target": row["target"],
                "target_idx": entity_mapping["targets"].get(row["target"], -1),
                "triplet_label": row["triplet_label"],
                "triplet_idx": triplet_to_idx.get(row["triplet_label"], -1),
            })
        
        phase = frame_df["phase"].iloc[0] if "phase" in frame_df.columns else "unknown"
        
        sequence_data.append({
            "frame": int(frame_id),
            "phase": phase,
            "num_active_triplets": len(active_indices),
            "active_triplet_indices": active_indices,
            "multi_hot": multi_hot.tolist(),
            "edges": edges,
        })
    
    with open(SEQ_DIR / f"{vid_name}_sequence.json", "w") as f:
        json.dump(sequence_data, f)
    
    print(f"  {vid_name}: {len(sequence_data)} frames")

print(f"\nAll sequences saved to {SEQ_DIR}")

## 5. Spot-Check: Sample Temporal Sequence

Pick a random video and inspect its first and last frame to verify correctness.

In [ ]:
sample_vid = "VID14"
with open(SEQ_DIR / f"{sample_vid}_sequence.json") as f:
    seq = json.load(f)

print(f"{sample_vid}: {len(seq)} frames\n")
print("First frame:")
print(json.dumps(seq[0], indent=2)[:600])
print(f"\nLast frame:")
print(json.dumps(seq[-1], indent=2)[:600])

# Verify multi-hot dimensions
assert len(seq[0]["multi_hot"]) == num_classes, "Multi-hot dimension mismatch!"
print(f"\nMulti-hot dimension: {len(seq[0]['multi_hot'])} (expected {num_classes}) OK")

## 6. Train / Val / Test Splits

70% train / 15% val / 15% test, seeded for reproducibility.

In [ ]:
all_videos = sorted(df_all["video"].unique().tolist())
np.random.seed(42)
shuffled = np.random.permutation(all_videos).tolist()

n = len(shuffled)
n_train = int(n * 0.70)
n_val = int(n * 0.15)

splits = {
    "train": sorted(shuffled[:n_train]),
    "val": sorted(shuffled[n_train:n_train + n_val]),
    "test": sorted(shuffled[n_train + n_val:]),
}

with open(OUTPUT_DIR / "data_splits.json", "w") as f:
    json.dump(splits, f, indent=2)

print(f"Train: {len(splits['train'])} videos")
print(f"Val:   {len(splits['val'])} videos")
print(f"Test:  {len(splits['test'])} videos")

# Show split details
for split_name, vids in splits.items():
    split_df = df_all[df_all["video"].isin(vids)]
    print(f"\n{split_name.upper()}: {len(vids)} videos, "
          f"{split_df['frame'].nunique()} unique frame IDs, "
          f"{len(split_df)} rows")
    print(f"  Videos: {vids}")

## 7. Week 1 Summary

| Output | Location |
|--------|----------|
| Per-video CSVs | `outputs/parsed_triplets/VIDxx_triplets.csv` |
| Combined CSV | `outputs/parsed_triplets/all_triplets.csv` |
| Graph files (GEXF) | `outputs/graphs/VIDxx_graph.gexf` |
| Statistics | `outputs/stats/` |
| Temporal sequences | `outputs/temporal_sequences/VIDxx_sequence.json` |
| Label mappings | `outputs/temporal_sequences/*_mapping.json` |
| Data splits | `outputs/data_splits.json` |

**Next: Week 2** - Build LSTM and Transformer baselines for next-action prediction.